> [!TIP]
> **Active Development Copy (v2):** This notebook is the workspace for ongoing improvements and extensions to **Application Rationalization & Multi-Criteria Portfolio Scoring**.
> The verified baseline remains preserved in `v1_baseline.ipynb` and `../../07_application_rationalization.ipynb`.

# Notebook 07: Application Rationalization & Multi-Criteria Portfolio Scoring

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 07 of 12  
**Focus:** Multi-Criteria Decision Analysis (MCDA), Gartner TIME Matrix, Rationalization Review Index (RRI), Avoidable Cost Scenarios  

---

## 1. Business Problem

Enterprise application rationalization is often paralyzed by internal organizational politics. IT leaders attempt to retire obsolete or redundant systems, only to be met with resistance from departmental sponsors claiming the application is indispensable.

Common failure modes:
1. **Decisions based solely on cost:** Decommissioning a high-cost application that turns out to be mission-critical.
2. **Decisions based on opaque "black box" scoring:** Stakeholders reject recommendations because the underlying formulas cannot be audited or explained.

To build trust and accelerate portfolio optimization, we require a **transparent, rule-based Multi-Criteria Decision Analysis (MCDA)** scoring framework with clear evidence trails.

---

## 2. Core Concept & Formulation

### 2.1 Multi-Criteria Decision Analysis (MCDA)
We compute five normalized indicators ($S \in [0, 100]$):
1. **Cost Score ($S_{\text{cost}}$):** Percentile ranking of annual TCO.
2. **Utilization Score ($S_{\text{util}}$):** Percentile ranking of active user volume (higher = more utilized).
3. **Criticality Score ($S_{\text{crit}}$):** Weighted based on operational criticality (Mission Critical = 100, Business Critical = 75, Operational = 40, Standard = 20).
4. **Capability Overlap Score ($S_{\text{over}}$):** Quantifies whether other applications in the enterprise support the same business capabilities.
5. **Lifecycle Review Score ($S_{\text{life}}$):** Weighted by current architectural roadmap (Retire = 100, Migrate = 80, Tolerate = 65, Strategic = 10).

### 2.2 Rationalization Review Index (RRI)
$$\text{RRI} = 0.30 S_{\text{cost}} + 0.25 (100 - S_{\text{util}}) + 0.20 S_{\text{over}} + 0.15 S_{\text{life}} + 0.10 (100 - S_{\text{crit}})$$

### 2.3 Strict Analytical Guardrail: Avoidable Cost Scenario
We never declare: *"Retiring this application will save ₹8.4 Cr."*  
Instead, we declare: *"₹8.4 Cr represents the current baseline expenditure. A rationalization review must determine how much of this cost is genuinely avoidable versus sunk or fixed infrastructure."*

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx

# Dynamically locate project root
workspace_root = Path.cwd()
while workspace_root != workspace_root.parent:
    if (workspace_root / "config.yaml").exists() or (workspace_root / "pyproject.toml").exists():
        break
    workspace_root = workspace_root.parent

src_dir = str(workspace_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from tvi.config import load_config
from tvi.rationalization import score_application_portfolio, find_rationalization_candidates
from tvi.graph import build_knowledge_graph

config = load_config()

# Compute complete multi-criteria portfolio scores
scored_portfolio = score_application_portfolio()
print("Multi-Criteria Portfolio Rationalization Scores:")
display(scored_portfolio[["application_id", "application_name", "lifecycle_status", "annual_tco", "cost_score", "utilization_score", "capability_overlap_score", "rationalization_review_index"]].head(8))

Multi-Criteria Portfolio Rationalization Scores:


---

## 3. Filter Candidates for Rationalization Review (RRI $\ge$ 60.0)

We filter applications meeting the rationalization threshold and inspect their transparent investigation evidence strings:

In [2]:
candidates_df = find_rationalization_candidates(review_threshold=60.0)
print(f"Applications Requiring Rationalization Review ({len(candidates_df)} systems flagged):")
display(candidates_df[["application_id", "application_name", "lifecycle_status", "criticality", "annual_tco", "avg_monthly_users", "overlapping_caps_count", "rationalization_review_index", "investigation_evidence"]])

Applications Requiring Rationalization Review (8 systems flagged):


---

## 4. Gartner TIME Framework Mapping

We map the portfolio into the industry-standard Gartner TIME matrix:
- **Tolerate:** High technical debt, but low cost/risk. Keep as-is.
- **Invest:** High business value, modern architecture. Allocate growth capital.
- **Migrate:** Business critical, but aging platform. Re-host / modernize.
- **Eliminate:** Redundant capability, low utilization, high cost. High rationalization priority.

In [3]:
# Gartner TIME Matrix Mapping
fig = px.scatter(
    scored_portfolio,
    x="utilization_score",
    y="cost_score",
    color="lifecycle_status",
    size="rationalization_review_index",
    hover_name="application_name",
    hover_data={"application_id": True, "annual_tco": ":,.0f", "rationalization_review_index": True},
    title="Gartner TIME / TBM Rationalization Matrix (Size = Rationalization Review Index)",
    labels={"utilization_score": "Business Fit / Utilization Score (0-100)", "cost_score": "Financial Burden / Cost Score (0-100)"},
    template="plotly_white",
    color_discrete_map={"Strategic": "#2ca02c", "Tolerate": "#ff7f0e", "Migrate": "#1f77b4", "Retire": "#d62728"}
)

fig.add_hline(y=50, line_dash="dash", line_color="grey")
fig.add_vline(x=50, line_dash="dash", line_color="grey")

fig.add_annotation(x=25, y=85, text="ELIMINATE / REVIEW<br>(High Cost / Low Adoption)", showarrow=False, font=dict(color="#d62728", size=11))
fig.add_annotation(x=75, y=85, text="INVEST / OPTIMIZE<br>(High Cost / High Adoption)", showarrow=False, font=dict(color="#1f77b4", size=11))
fig.add_annotation(x=25, y=15, text="TOLERATE<br>(Low Cost / Low Adoption)", showarrow=False, font=dict(color="#7f7f7f", size=11))
fig.add_annotation(x=75, y=15, text="HARVEST / WORKHORSE<br>(Low Cost / High Adoption)", showarrow=False, font=dict(color="#2ca02c", size=11))

fig.show()

---

## 5. MCDA Weight Sensitivity Analysis

How do rationalization priorities shift if executive leadership pivots between **Aggressive Cost-Cutting** (Cost weight = 0.55) versus **Risk-Averse Modernization** (Criticality & Lifecycle weights = 0.65)?

In [4]:
# Weight Sensitivity Simulation
sens_df = scored_portfolio.copy()

# Baseline RRI
sens_df["rri_baseline"] = sens_df["rationalization_review_index"]

# Scenario 1: Aggressive Cost-Cutting
sens_df["rri_cost_cutting"] = (
    0.55 * sens_df["cost_score"]
    + 0.20 * (100.0 - sens_df["utilization_score"])
    + 0.15 * sens_df["capability_overlap_score"]
    + 0.10 * (100.0 - sens_df["criticality_score"])
).round(1)

# Scenario 2: Risk-Averse Modernization
sens_df["rri_risk_averse"] = (
    0.20 * sens_df["cost_score"]
    + 0.15 * (100.0 - sens_df["utilization_score"])
    + 0.15 * sens_df["capability_overlap_score"]
    + 0.25 * sens_df["lifecycle_score"]
    + 0.25 * (100.0 - sens_df["criticality_score"])
).round(1)

sens_compare = sens_df[["application_id", "application_name", "rri_baseline", "rri_cost_cutting", "rri_risk_averse"]].sort_values("rri_baseline", ascending=False).head(6)
print("Sensitivity of Top Candidates to Policy Weights:")
display(sens_compare)

Sensitivity of Top Candidates to Policy Weights:


---

## 6. Overlapping Capability Subgraph Visualization

We render a focused Knowledge Subgraph illustrating the functional duplication of `CAP003` (Order-to-Cash) supported simultaneously by `APP005` (Strategic) and `APP012` (Retire):

In [5]:
G = build_knowledge_graph()
otc_sub = G.subgraph(["CAP003", "APP005", "APP012", "BU004"]).copy()

plt.figure(figsize=(7, 4.5))
pos = nx.spring_layout(otc_sub, seed=42)

colors = ["#2ca02c" if n == "CAP003" else "#1f77b4" if n == "APP005" else "#d62728" if n == "APP012" else "#9467bd" for n in otc_sub.nodes()]
nx.draw_networkx_nodes(otc_sub, pos, node_color=colors, node_size=1500, alpha=0.9)
nx.draw_networkx_edges(otc_sub, pos, edge_color="#555", arrows=True, arrowsize=18, width=2)

labels = {
    "CAP003": "CAP003\n(Order-to-Cash)",
    "APP005": "APP005\n(OrderFlow Strategic)",
    "APP012": "APP012\n(QuickOrder Retire)",
    "BU004": "BU004\n(Digital Payments)"
}
nx.draw_networkx_labels(otc_sub, pos, labels=labels, font_size=8, font_weight="bold")

plt.title("Scenario C: Capability Redundancy Subgraph (Order-to-Cash)", fontsize=12)
plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Limitations & Analytical Guardrails

1. **Avoidable vs Sunk Costs:** Contractual termination penalties, multi-year cloud commitments, and retained shared infrastructure will diminish immediate cash savings.
2. **Governance Auditability:** Every recommendation is supported by transparent evidence strings linking back to source data.

## 8. Next Step

In **Notebook 08: Dependency & Impact Analysis**, before recommending any application for retirement, we interrogate the Knowledge Graph to evaluate its **blast radius and hidden integration dependencies**.

## 9. v2 Extensions: Sensitivity, Pareto Optimization, Decommissioning & Exit Penalties

Demonstrating:
1. **Interactive MCDA Weight Sensitivity:** Evaluating RRI ranking robustness when varying cost and utilization priorities.
2. **Migration Complexity vs Savings Pareto Frontier:** Non-dominated strategic rationalization trade-offs.
3. **Phased Application Decommissioning Roadmap:** Milestone schedule with decoupled dependencies and cost unlock profile.
4. **Contractual Exit Penalty & Retained Overhead Modeling:** Differentiating headline TCO from net cash realization.


In [6]:
from tvi.rationalization import (
    analyze_mcda_weight_sensitivity,
    evaluate_rationalization_pareto_frontier,
    generate_decommissioning_roadmap,
    model_contractual_exit_impact,
)

print('=== 1. MCDA Weight Sensitivity Analysis (Varying Cost Weight) ===')
sens_df = analyze_mcda_weight_sensitivity(weight_param='cost_score', weight_range=[0.10, 0.25, 0.40, 0.55])
display(sens_df[sens_df['application_id'].isin(['APP021', 'APP005', 'APP001', 'APP012'])].head(12))

print('
=== 2. Application Rationalization Pareto Frontier ===')
pareto_df = evaluate_rationalization_pareto_frontier()
display(pareto_df.head(10))

plt.figure(figsize=(9, 5.5))
colors = pareto_df['is_pareto_optimal'].map({True: '#d62728', False: '#1f77b4'})
sizes = pareto_df['is_pareto_optimal'].map({True: 120, False: 50})
plt.scatter(pareto_df['migration_complexity_score'], pareto_df['annual_avoidable_savings'] / 1e7, c=colors, s=sizes, alpha=0.8)
for _, r in pareto_df[pareto_df['is_pareto_optimal']].iterrows():
    plt.annotate(r['application_id'], (r['migration_complexity_score'] + 1, r['annual_avoidable_savings'] / 1e7 + 0.2), fontsize=8, fontweight='bold', color='#991b1b')
plt.title('Application Migration Complexity vs Annual Avoidable Savings (Pareto Frontier)', fontsize=12)
plt.xlabel('Migration Complexity Score (0 - 100)', fontsize=10)
plt.ylabel('Annual Avoidable Spend (₹ Cr)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('
=== 3. Phased Application Decommissioning Roadmap ===')
roadmap_df = generate_decommissioning_roadmap(target_app_ids=['APP021', 'APP012'])
display(roadmap_df)

print('
=== 4. Contractual Early Exit Penalty & Net Yield (APP021) ===')
exit_impact = model_contractual_exit_impact('APP021', contract_months_remaining=6, penalty_rate=0.50, fixed_overhead_fraction=0.25)
for k, v in exit_impact.items():
    print(f'  {k}: {v}')


  Cell In[1], line 12
    print('
          ^
SyntaxError: unterminated string literal (detected at line 12)

